# EDA

In [23]:
import os
import urllib
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env", override=True)

server = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")
driver = os.getenv("SQL_DRIVER")
user = os.getenv("SQL_USER")
password = os.getenv("SQL_PASSWORD")

print("Server:", server)
print("Database:", database)
print("Driver:", driver)


Server: 172.24.112.1
Database: ecommerce_xomdata
Driver: ODBC Driver 18 for SQL Server


## 1. Load Raw Data from SQL Server

In [62]:
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=15;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    pool_pre_ping=True
)

with engine.connect() as conn:
    print("Kết nối SQL Server thành công!")

Kết nối SQL Server thành công!


In [26]:
sql_tables = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND TABLE_SCHEMA = 'e_commerce'
"""

tables = pd.read_sql(sql_tables, engine)

print(tables)

               TABLE_NAME
0            dim_customer
1              fact_sales
2             dim_product
3              dim_region
4            rfm_features
5           market_basket
6                  cohort
7   customer_segmentation
8                customer
9              ecom_sales
10                product
11                 region


In [27]:
df_customer = pd.read_sql("SELECT * FROM e_commerce.customer", engine)
df_sales = pd.read_sql("SELECT * FROM e_commerce.ecom_sales", engine)
df_product = pd.read_sql("SELECT * FROM e_commerce.product", engine)
df_region = pd.read_sql("SELECT * FROM e_commerce.region", engine)

## 2. Dataset Overview

* Kiểm tra head(), info(), describe() -> Missing data, unique code hay không, null value
* Kiếm tra giá trị duplicate
* Category Consistency

### 2.1. Sales Table

In [28]:
df_sales.head()

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit
0,1,MX-2015-SC2057582-42279,2023-10-02,SC-2057582,Consumer,R2140,P000409,3,48.0,0.0,19.2
1,2,MX-2013-KW1657028-41562,2021-10-15,KW-1657028,Consumer,R0816,P000147,8,48.0,0.0,7.2
2,3,MX-2013-KW1657028-41562,2021-10-15,KW-1657028,Consumer,R0816,P000395,2,16.0,0.0,8.0
3,4,MX-2013-KW1657028-41562,2021-10-15,KW-1657028,Consumer,R0816,P002140,4,28.0,0.0,14.0
4,5,MX-2013-KW1657028-41562,2021-10-15,KW-1657028,Consumer,R0816,P002828,2,20.0,0.0,10.0


In [29]:
df_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   row_id        51290 non-null  int64  
 1   order_id      51290 non-null  str    
 2   order_date    51290 non-null  object 
 3   customer_id   51290 non-null  str    
 4   segment       51290 non-null  str    
 5   region_code   51290 non-null  str    
 6   product_code  51290 non-null  str    
 7   quantity      51290 non-null  int64  
 8   sales         51290 non-null  float64
 9   discount      51290 non-null  float64
 10  profit        51290 non-null  float64
dtypes: float64(3), int64(2), object(1), str(5)
memory usage: 4.3+ MB


In [30]:
df_sales.describe(include='all')

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit
count,51290.00000,51290,51290,51290,51290,51290,51290,51290.000000,51290.000000,51290.000000,51290.000000
unique,NaN,25728,1430,17415,3,3829,3578,NaN,NaN,NaN,NaN
top,NaN,CA-2015-SV20365140-42268,2023-06-18,SV-203651406,Consumer,R3174,P000116,NaN,NaN,NaN,NaN
freq,NaN,14,135,26,26518,915,335,NaN,NaN,NaN,NaN
mean,25645.50000,NaN,NaN,NaN,NaN,NaN,NaN,5.415832,127.074946,0.142883,20.772345
std,14806.29199,NaN,NaN,NaN,NaN,NaN,NaN,4.908234,236.074764,0.212285,83.582681
min,1.00000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,2.000000,0.000000,-1746.000000
25%,12823.25000,NaN,NaN,NaN,NaN,NaN,NaN,2.000000,25.000000,0.000000,0.000000
50%,25645.50000,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,56.000000,0.000000,7.200000
75%,38467.75000,NaN,NaN,NaN,NaN,NaN,NaN,7.000000,132.000000,0.200000,25.000000


In [31]:
df_sales['segment'].value_counts(dropna=False)

segment
Consumer         26518
Corporate        15429
Self-Employed     9343
Name: count, dtype: int64

In [32]:
df_skew = df_sales['sales'].skew()
print(f'Skew: {df_skew}')
def iqr_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data < lower) | (data > upper)]
    return outliers, Q1, Q3, IQR, lower, upper
df_outliers, q1, q3, iqr, lower, upper = iqr_outliers(df_sales['sales'])

print(f'25% :{q1}')
print(f'75% :{q3}')
print(f'IQR :{iqr}')
print(f'Lower quartile :{lower}')
print(f'Upper quartile :{upper}')

Skew: 6.402469124156644
25% :25.0
75% :132.0
IQR :107.0
Lower quartile :-135.5
Upper quartile :292.5


### 2.2 Customer Table

In [33]:
df_customer.head()

,customer_id,first_name,last_name,birth_date,marital_status,gender,email_address,annual_income,education_level,occupation,home_owner
0,AA-10315102,Bryant,Vazquez,1980-01-21,M,M,bryant6@hotmail.com,20000,Bachelors,Clerical,Y
1,AA-10315120,Jon,Rubio,1998-12-01,S,M,jon6@hotmail.com,20000,Bachelors,Clerical,N
2,AA-10315139,Jésus,Hernandez,1973-01-27,M,M,jésus20@hotmail.com,30000,Bachelors,Clerical,Y
3,AA-103151402,Marcus,Taylor,1999-01-24,S,M,marcus31@hotmail.com,40000,Partial College,Skilled Manual,N
4,AA-103151404,Jesse,Dominguez,1968-09-11,S,M,jesse34@hotmail.com,40000,High School,Skilled Manual,Y


In [34]:
df_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 17415 entries, 0 to 17414
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   customer_id      17415 non-null  str   
 1   first_name       17415 non-null  str   
 2   last_name        17415 non-null  str   
 3   birth_date       17415 non-null  object
 4   marital_status   17415 non-null  str   
 5   gender           17291 non-null  str   
 6   email_address    17415 non-null  str   
 7   annual_income    17415 non-null  int64 
 8   education_level  17415 non-null  str   
 9   occupation       17415 non-null  str   
 10  home_owner       17415 non-null  str   
dtypes: int64(1), object(1), str(9)
memory usage: 1.5+ MB


In [35]:
df_customer.describe(include='all')

,customer_id,first_name,last_name,birth_date,marital_status,gender,email_address,annual_income,education_level,occupation,home_owner
count,17415,17415,17415,17415,17415,17291,17415,17415.000000,17415,17415,17415
unique,17415,663,370,9893,2,2,17415,NaN,5,5,2
top,AA-10315102,Marcus,Diaz,1965-10-07,M,M,bryant6@hotmail.com,NaN,Bachelors,Professional,Y
freq,1,93,198,8,9491,8733,1,NaN,5036,5176,11850
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57190.927361,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32213.237506,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000.000000,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30000.000000,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60000.000000,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70000.000000,NaN,NaN,NaN


In [36]:
df_customer.duplicated().sum()

np.int64(0)

In [37]:
categorical_cols = [
    'marital_status',
    'gender',
    'education_level',
    'occupation',
    'home_owner'
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(df_customer[col].value_counts(dropna=False))


===== marital_status =====
marital_status
M    9491
S    7924
Name: count, dtype: int64

===== gender =====
gender
M      8733
F      8558
NaN     124
Name: count, dtype: int64

===== education_level =====
education_level
Bachelors              5036
Partial College        4780
High School            3107
Graduate Degree        3014
Partial High School    1478
Name: count, dtype: int64

===== occupation =====
occupation
Professional      5176
Skilled Manual    4321
Management        2887
Clerical          2750
Manual            2281
Name: count, dtype: int64

===== home_owner =====
home_owner
Y    11850
N     5565
Name: count, dtype: int64


### 2.3 Product Table

In [38]:
df_product.head()

,product_code,product,category,subcategory
0,P000001,Soap & Glory Endless Glove Moisturizing Hand C...,Body care,"bath oils, bubbles and soaks"
1,P000002,Pantene Pro-V Truly Relaxed Hair Lightweight C...,Body care,"bath oils, bubbles and soaks"
2,P000003,Estée Lauder Pure Color Envy Irresistible,Body care,"bath oils, bubbles and soaks"
3,P000004,Gold Infinity Bracelet,Body care,body moisturizers
4,P000005,Deborah Lippmann Nail Polish Lady is a Tramp C...,Body care,body moisturizers


In [39]:
df_product.info()

<class 'pandas.DataFrame'>
RangeIndex: 28356 entries, 0 to 28355
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_code  28356 non-null  str  
 1   product       28356 non-null  str  
 2   category      28356 non-null  str  
 3   subcategory   28356 non-null  str  
dtypes: str(4)
memory usage: 886.3 KB


In [40]:
df_product.describe(include='all')

,product_code,product,category,subcategory
count,28356,28356,28356,28356
unique,28356,3577,5,17
top,P000001,NARS Single Eyeshadow Persia Matte Paprika,Body care,Nail care products
freq,1,17,11548,2386


In [41]:
df_product.duplicated().sum()

np.int64(0)

In [42]:
categorical_cols = [
    'category',
    'subcategory'
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(df_product[col].value_counts(dropna=False))


===== category =====
category
Body care               11548
Home and Accessories     5912
Make up                  4941
Hair care                4070
Face care                1885
Name: count, dtype: int64

===== subcategory =====
subcategory
Nail care products              2386
shampoos and conditioners       2259
Eye shadows and pencils         2178
vitamins and supplements        1929
brushes and applicators         1913
face moisturizing products      1885
hair colors and toners          1811
Accessories                     1797
hand creams                     1577
body moisturizers               1551
foundations and concealers      1529
face masks and exfoliators      1498
bath oils, bubbles and soaks    1474
candles, sprays, diffusers      1464
Lipsticks                       1234
Body soaps and washes           1133
fragrances                       738
Name: count, dtype: int64


### 2.4 Region Table

In [43]:
df_region.head()

,region_code,city,state,country,country_latitude,country_longitude,region,market
0,R0001,Jalalabad,Nangarhar,Afghanistan,33.939110,67.709953,Southern Asia,Asia Pacific
1,R0002,Herat,Hirat,Afghanistan,33.939110,67.709953,Southern Asia,Asia Pacific
2,R0003,Kabul,Kabul,Afghanistan,33.939110,67.709953,Southern Asia,Asia Pacific
3,R0004,Kandahar,Kandahar,Afghanistan,33.939110,67.709953,Southern Asia,Asia Pacific
4,R0005,Elbasan,Elbasan,Albania,41.153332,20.168331,Southern Europe,Europe


In [44]:
df_region.info()

<class 'pandas.DataFrame'>
RangeIndex: 3828 entries, 0 to 3827
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   region_code        3828 non-null   str    
 1   city               3828 non-null   str    
 2   state              3828 non-null   str    
 3   country            3828 non-null   str    
 4   country_latitude   3828 non-null   float64
 5   country_longitude  3828 non-null   float64
 6   region             3828 non-null   str    
 7   market             3828 non-null   str    
dtypes: float64(2), str(6)
memory usage: 239.4 KB


In [45]:
df_region.describe(include='all')

,region_code,city,state,country,country_latitude,country_longitude,region,market
count,3828,3828,3828,3828,3828.000000,3828.000000,3828,3828
unique,3828,3650,1102,164,NaN,NaN,23,5
top,R0001,Burlington,England,United States,NaN,NaN,Western Europe,Europe
freq,1,4,149,604,NaN,NaN,653,1281
mean,NaN,NaN,NaN,NaN,28.510730,-1.314251,NaN,NaN
std,NaN,NaN,NaN,NaN,24.018286,70.745155,NaN,NaN
min,NaN,NaN,NaN,NaN,-40.900557,-106.346771,NaN,NaN
25%,NaN,NaN,NaN,NaN,20.593684,-66.589730,NaN,NaN
50%,NaN,NaN,NaN,NaN,37.090240,2.213749,NaN,NaN
75%,NaN,NaN,NaN,NaN,46.227638,35.243322,NaN,NaN


In [46]:
df_region.duplicated().sum()

np.int64(0)

In [47]:
categorical_cols = [
    'market',
    'region'
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(df_region[col].value_counts(dropna=False))


===== market =====
market
Europe          1281
Asia Pacific     931
USCA             650
LATAM            645
Africa           321
Name: count, dtype: int64

===== region =====
region
Western Europe       653
South America        435
Eastern Asia         332
Southern Asia        241
Southern Europe      238
Northern Europe      208
Central US           186
Eastern Europe       182
Western US           170
Central America      166
Western Asia         157
Southern US          134
Eastern US           114
Oceania               93
Western Africa        85
Southeastern Asia     84
North Africa          81
Eastern Africa        74
Canada                46
Caribbean             44
Central Africa        42
Southern Africa       39
Central Asia          24
Name: count, dtype: int64


## 3. Data Type Conversion

In [48]:
# Convert order_date từ text sang datetime
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# Convert birth_date
df_customer['birth_date'] = pd.to_datetime(df_customer['birth_date'])

## 4. Missing Values Recheck

In [49]:
df_customer['gender'] = df_customer['gender'].fillna('Unknown')

In [50]:
df_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 17415 entries, 0 to 17414
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   customer_id      17415 non-null  str          
 1   first_name       17415 non-null  str          
 2   last_name        17415 non-null  str          
 3   birth_date       17415 non-null  datetime64[s]
 4   marital_status   17415 non-null  str          
 5   gender           17415 non-null  str          
 6   email_address    17415 non-null  str          
 7   annual_income    17415 non-null  int64        
 8   education_level  17415 non-null  str          
 9   occupation       17415 non-null  str          
 10  home_owner       17415 non-null  str          
dtypes: datetime64[s](1), int64(1), str(9)
memory usage: 1.5 MB


## 5. Referential Integrity

Kiếm tra xem thông tin customer, product, region trong bảng Sales có trong Customer, Product, Region hay không?

### 5.1 Check referential Integrity

In [51]:
sales_customers = set(df_sales['customer_id'].dropna())
master_customers = set(df_customer['customer_id'].dropna())

missing_customers = sales_customers - master_customers

print(f'Customers in sales but missing from customer master: {len(missing_customers)}')
print(missing_customers)

Customers in sales but missing from customer master: 0
set()


In [52]:
sales_products = set(df_sales['product_code'].dropna())
master_products = set(df_product['product_code'].dropna())

missing_products = sales_products - master_products

print(f'Products in sales but missing from product master: {len(missing_products)}')
print(missing_products)

Products in sales but missing from product master: 2
{'PPP000002', 'PPP000010'}


In [53]:
sales_regions = set(df_sales['region_code'].dropna())
master_regions = set(df_region['region_code'].dropna())

missing_regions = sales_regions - master_regions

print(f'regions in sales but missing from region master: {len(missing_regions)}')
print(missing_regions)

regions in sales but missing from region master: 1
{'RRR0001'}


In [54]:
df_sales[df_sales['product_code'].isin(missing_products)]

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit
25219,25220,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,RRR0001,PPP000010,5,70.0,0.0,35.0
29463,29464,ID-2013-BD116051-41518,2021-09-01,BD-116051,Consumer,R0002,PPP000002,4,20.0,0.0,10.0


### 5.2 Xử lý referential Integrity

#### 5.2.1. Product

In [55]:
import re

invalid_format = df_sales[
    ~df_sales['product_code'].astype(str).str.match(r'^P\d{6}$', na=False)
]

invalid_format['product_code'].value_counts()

product_code
PPP000010    1
PPP000002    1
Name: count, dtype: int64

Hai code sản phẩm không có trong bảng Product có khả năng là viết nhầm syntax (vì syntax chung của code sản phẩm là Pxxxxxx) inconsistent product code/mapping issue. => Để đảm bảo kết luận này đúng thì sẽ check xem thử giá của các sản phẩm PPP000010 vs P000010 và PPP000002 vs P000002 có giống nhau hay không

In [56]:
df_check1 = df_sales[df_sales['product_code'].isin(['PPP000010','P000010'])]
df_check1['price'] = df_check1['sales']/df_check1['quantity']
df_check1

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit,price
2121,2122,MX-2014-JM1558028-41734,2022-04-05,JM-1558028,Consumer,R0816,P000010,3,42.0,0.0,21.0,14.0
3197,3198,MX-2014-PO1918018-41907,2022-09-25,PO-1918018,Self-Employed,R0247,P000010,4,56.0,0.0,28.0,14.0
4754,4755,MX-2012-PG1882082-41188,2020-10-06,PG-1882082,Consumer,R2145,P000010,5,70.0,0.0,35.0,14.0
13812,13813,ES-2014-PM1913545-41909,2022-09-27,PM-1913545,Self-Employed,R1161,P000010,3,42.0,0.0,21.0,14.0
16096,16097,ES-2012-DK1289564-41045,2020-05-16,DK-1289564,Consumer,R1885,P000010,2,28.0,0.0,14.0,14.0
21663,21664,IN-2012-ES1408027-40941,2020-02-02,ES-1408027,Corporate,R0671,P000010,3,42.0,0.0,21.0,14.0
25219,25220,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,RRR0001,PPP000010,5,70.0,0.0,35.0,14.0
28619,28620,IN-2015-DN1369027-42350,2023-12-12,DN-1369027,Consumer,R0740,P000010,6,84.0,0.0,33.6,14.0
28990,28991,IN-2015-JJ1544558-42020,2023-01-16,JJ-1544558,Consumer,R1640,P000010,3,42.0,0.0,16.8,14.0
31413,31414,CA-2013-DK13090140-41335,2021-03-02,DK-130901404,Consumer,R3216,P000010,3,42.0,0.0,4.2,14.0


In [57]:
df_check2 = df_sales[df_sales['product_code'].isin(['PPP000002','P000002'])]
df_check2['price'] = df_check2['sales']/df_check2['quantity']
df_check2

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit,price
246,247,MX-2012-AC1042031-41136,2020-08-15,AC-1042031,Corporate,R0890,P000002,15,75.0,0.00,7.50,5.0
508,509,MX-2013-BF1121593-41541,2021-09-24,BF-1121593,Self-Employed,R2378,P000002,4,20.0,0.00,1.96,5.0
981,982,MX-2015-AS10090101-42118,2023-04-24,AS-10090101,Consumer,R2475,P000002,2,10.0,0.40,-3.00,5.0
2937,2938,MX-2013-AM1036018-41578,2021-10-31,AM-1036018,Corporate,R0290,P000002,5,25.0,0.00,12.50,5.0
4453,4454,MX-2015-TG2131082-42208,2023-07-23,TG-2131082,Consumer,R2164,P000002,4,20.0,0.00,2.00,5.0
6752,6753,MX-2012-KM1637518-41074,2020-06-14,KM-1637518,Self-Employed,R0348,P000002,1,5.0,0.00,2.50,5.0
9491,9492,MX-2014-SP2054582-41992,2022-12-19,SP-2054582,Corporate,R2173,P000002,15,75.0,0.00,7.50,5.0
9604,9605,MX-2014-CP1234018-41804,2022-06-14,CP-1234018,Corporate,R0291,P000002,6,30.0,0.00,3.00,5.0
10315,10316,ES-2015-DR1288048-42173,2023-06-18,DR-1288048,Corporate,R1518,P000002,18,90.0,0.10,27.00,5.0
11655,11656,ES-2012-TS211608-41074,2020-06-14,TS-211608,Corporate,R0136,P000002,2,10.0,0.00,1.00,5.0


Sau khi tính giá thì cho thấyPPP000010 vs P000010 và PPP000002 vs P000002 có giá bằng nhau => inconsistent product code/mapping issue

In [58]:
df_sales['product_code'] = df_sales['product_code'].replace(
    {'PPP000010': 'P000010','PPP000002': 'P000002' }
)

#### 5.2.2. Region

In [59]:
import re

invalid_format = df_sales[
    ~df_sales['region_code'].astype(str).str.match(r'^R\d{4}$', na=False)
]

invalid_format['region_code'].value_counts()

region_code
RRR0001    2
Name: count, dtype: int64

In [60]:
df_check3 = df_sales[df_sales['region_code'].isin(['R0001','RRR0001'])]
df_check3

,row_id,order_id,order_date,customer_id,segment,region_code,product_code,quantity,sales,discount,profit
25216,25217,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,R0001,P000044,12,132.0,0.0,13.2
25217,25218,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,R0001,P000013,18,216.0,0.0,108.0
25218,25219,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,R0001,P000049,2,34.0,0.0,17.0
25219,25220,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,RRR0001,P000010,5,70.0,0.0,35.0
25220,25221,IN-2012-TH211151-41003,2020-04-04,TH-211151,Corporate,RRR0001,P000001,13,416.0,0.0,208.0


RRR0001 và R0001 cho ra cùng một khách hàng => Sai syntax

In [61]:
df_sales['region_code'] = df_sales['region_code'].replace(
    {'RRR0001': 'R0001'}
)